In [ ]:
import matplotlib.pyplot as plt
import pickle
import numpy as np
from pathlib import Path

legend_properties = {'weight': 'bold'}

In [ ]:
# Data folder
# Mac path for the Hybrid experiment on re-centralization degree.
data_folder = Path(r"/Volumes/T7/data/dao-0310-23/V4_4/Hybrid_2")

# The revised hybrid_run.py saves the focal parameter as
# "hybrid_recentralization_list". The fallback keeps the notebook compatible
# with older result folders that still use "hybrid_beta_list".
recentralization_list_file = data_folder / "hybrid_recentralization_list"
legacy_beta_list_file = data_folder / "hybrid_beta_list"

performance_file = data_folder / "hybrid_performance"
diversity_file = data_folder / "hybrid_diversity"
variance_file = data_folder / "hybrid_variance"
cv_file = data_folder / "hybrid_cv"
consensus_performance_file = data_folder / "hybrid_consensus_performance"
entropy_file = data_folder / "hybrid_entropy"
antagonism_file = data_folder / "hybrid_antagonism"
mode_file = data_folder / "hybrid_mode"

if recentralization_list_file.exists():
    with open(recentralization_list_file, 'rb') as infile:
        recentralization_list = pickle.load(infile)
else:
    with open(legacy_beta_list_file, 'rb') as infile:
        recentralization_list = pickle.load(infile)

with open(performance_file, 'rb') as infile:
    hybrid_performance = pickle.load(infile)
with open(diversity_file, 'rb') as infile:
    hybrid_diversity = pickle.load(infile)
with open(variance_file, 'rb') as infile:
    hybrid_variance = pickle.load(infile)
with open(cv_file, 'rb') as infile:
    hybrid_cv = pickle.load(infile)
with open(consensus_performance_file, 'rb') as infile:
    hybrid_consensus_performance = pickle.load(infile)
with open(entropy_file, 'rb') as infile:
    hybrid_entropy = pickle.load(infile)
with open(antagonism_file, 'rb') as infile:
    hybrid_antagonism = pickle.load(infile)

# Mode is diagnostic; it is large and not necessary for core plots.
# Load it only when needed.
try:
    with open(mode_file, 'rb') as infile:
        hybrid_mode = pickle.load(infile)
except FileNotFoundError:
    hybrid_mode = None

# Convert to arrays: shape = [len(recentralization_list), search_loop]
hybrid_performance = np.array(hybrid_performance)
hybrid_diversity = np.array(hybrid_diversity)
hybrid_variance = np.array(hybrid_variance)
hybrid_cv = np.array(hybrid_cv)
hybrid_consensus_performance = np.array(hybrid_consensus_performance)
hybrid_entropy = np.array(hybrid_entropy)
hybrid_antagonism = np.array(hybrid_antagonism)
recentralization_array = np.array(recentralization_list)

print("recentralization_list:", recentralization_list)
print("performance shape:", hybrid_performance.shape)
print("diversity shape:", hybrid_diversity.shape)
print("variance shape:", hybrid_variance.shape)
print("consensus performance shape:", hybrid_consensus_performance.shape)


In [ ]:
# Define more colors
# NUS two colors
nus_blue = "#003D7C"
nus_orange = "#EF7C00"
# Nature three colors
nature_orange = "#F16C23"
nature_blue = "#2B6A99"
nature_green = "#1B7C3D"
# Morandi six colors
morandi_blue = "#046586"
morandi_green =  "#28A9A1"
morandi_yellow = "#C9A77C"
morandi_orange = "#F4A016"
morandi_pink = "#F6BBC6"
morandi_red = "#E71F19"
morandi_purple = "#B08BEB"
# Others
shallow_grey = "#D3D4D3"
deep_grey = "#A6ABB6"

In [ ]:
# Helper functions

def set_axis_style(ax):
    ax.spines["left"].set_linewidth(1.5)
    ax.spines["right"].set_linewidth(1.5)
    ax.spines["top"].set_linewidth(1.5)
    ax.spines["bottom"].set_linewidth(1.5)


def final_value(matrix):
    """Return final-period value for each beta."""
    return np.array(matrix)[:, -1]


def window_average(matrix, window=50):
    """Return average over the last `window` periods for each beta."""
    matrix = np.array(matrix)
    return np.mean(matrix[:, -window:], axis=1)


def savefig(name):
    plt.savefig(data_folder / name, transparent=True, dpi=300, bbox_inches='tight')

In [ ]:
# Impact of re-centralization on final performance
# In the revised Hybrid.py, beta directly captures the degree of re-centralization.
# beta = 0: fully decentralized consensus channel
# beta = 1: fully hierarchical specification channel

order = np.argsort(recentralization_array)

x = recentralization_array[order]
individual_final = np.array(final_value(hybrid_performance))[order]
consensus_final = np.array(final_value(hybrid_consensus_performance))[order]

fig, ax = plt.subplots()
set_axis_style(ax)

ax.plot(x, individual_final, "-s", color=nature_orange,
        label="Individual performance")
ax.plot(x, consensus_final, "-v", color=nature_blue,
        label="Consensus performance")

plt.xlabel(r'Re-centralization Degree ($\beta$)', fontweight='bold', fontsize=12)
plt.ylabel('Final Performance', fontweight='bold', fontsize=12)
plt.xticks(x)
ax.legend(frameon=False, ncol=1, fontsize=12, prop=legend_properties)

savefig("hybrid_recentralization_final_performance.png")
plt.show()
plt.clf()

print("re-centralization list:", x.tolist())
print("performance list:", individual_final.tolist())
print("consensus performance list:", consensus_final.tolist())


In [ ]:
# Impact of re-centralization on late-stage average performance
# This plot is less sensitive to final-period random fluctuation than the final-value plot.
window = 50
fig, ax = plt.subplots()
set_axis_style(ax)

ax.plot(recentralization_array, window_average(hybrid_performance, window=window), "-s",
        color=nature_orange, label="Individual performance")
ax.plot(recentralization_array, window_average(hybrid_consensus_performance, window=window), "-v",
        color=nature_blue, label="Consensus performance")

plt.xlabel(r'Re-centralization Degree ($\beta$)', fontweight='bold', fontsize=12)
plt.ylabel('Late-stage Average Performance', fontweight='bold', fontsize=12)
plt.xticks(recentralization_array)
ax.legend(frameon=False, ncol=1, fontsize=12, prop=legend_properties)
savefig("hybrid_recentralization_late_stage_performance.png")
plt.show()
plt.clf()

# Interpretation note:
# Light re-centralization may create coordination interference before generating
# coordination benefits. Limited top-down specification can compress belief
# disparity and interrupt decentralized exploration before hierarchical alignment
# becomes strong enough to improve coordinated adaptation.


In [ ]:
# Impact of re-centralization on diversity
fig, ax = plt.subplots()
set_axis_style(ax)

ax.plot(recentralization_array, final_value(hybrid_diversity), "-s",
        color=nature_green, label="Diversity")

plt.xlabel(r'Re-centralization Degree ($\beta$)', fontweight='bold', fontsize=12)
plt.ylabel('Final Diversity', fontweight='bold', fontsize=12)
plt.xticks(recentralization_array)
ax.legend(frameon=False, ncol=1, fontsize=12, prop=legend_properties)
savefig("hybrid_recentralization_final_diversity.png")
plt.show()
plt.clf()


In [ ]:
# Impact of re-centralization on variance
fig, ax = plt.subplots()
set_axis_style(ax)

ax.plot(recentralization_array, final_value(hybrid_variance), "-s",
        color=nature_blue, label="Std. Dev. of individual knowledge levels")

plt.xlabel(r'Re-centralization Degree ($\beta$)', fontweight='bold', fontsize=12)
plt.ylabel('Final Std. Dev. of Individual Knowledge Levels', fontweight='bold', fontsize=9)
plt.xticks(recentralization_array)
ax.legend(frameon=False, ncol=1, fontsize=12, prop=legend_properties)
savefig("hybrid_recentralization_final_sd.png")
plt.show()
plt.clf()


In [ ]:
# Impact of re-centralization on coefficient of variation
fig, ax = plt.subplots()
set_axis_style(ax)

ax.plot(recentralization_array, final_value(hybrid_cv), "-s",
        color=nature_orange, label="Coefficient of variation")

plt.xlabel(r'Re-centralization Degree ($\beta$)', fontweight='bold', fontsize=12)
plt.ylabel('Final Coefficient of Variation', fontweight='bold', fontsize=12)
plt.xticks(recentralization_array)
ax.legend(frameon=False, ncol=1, fontsize=12, prop=legend_properties)
savefig("hybrid_recentralization_final_cv.png")
plt.show()
plt.clf()


In [ ]:
# Impact of re-centralization on entropy
fig, ax = plt.subplots()
set_axis_style(ax)

ax.plot(recentralization_array, final_value(hybrid_entropy), "-s",
        color=nature_green, label="Entropy")

plt.xlabel(r'Re-centralization Degree ($\beta$)', fontweight='bold', fontsize=12)
plt.ylabel('Final Entropy', fontweight='bold', fontsize=12)
plt.xticks(recentralization_array)
ax.legend(frameon=False, ncol=1, fontsize=12, prop=legend_properties)
savefig("hybrid_recentralization_final_entropy.png")
plt.show()
plt.clf()


In [ ]:
# Impact of re-centralization on antagonism
fig, ax = plt.subplots()
set_axis_style(ax)

ax.plot(recentralization_array, final_value(hybrid_antagonism), "-s",
        color=nature_orange, label="Antagonism")

plt.xlabel(r'Re-centralization Degree ($\beta$)', fontweight='bold', fontsize=12)
plt.ylabel('Final Antagonism', fontweight='bold', fontsize=12)
plt.xticks(recentralization_array)
ax.legend(frameon=False, ncol=1, fontsize=12, prop=legend_properties)
savefig("hybrid_recentralization_final_antagonism.png")
plt.show()
plt.clf()


In [ ]:
# Time trajectories by selected re-centralization values
selected_recentralization_values = [0.0, 0.3, 0.5, 0.7, 1.0]
selected_indices = [
    int(np.argmin(np.abs(recentralization_array - value)))
    for value in selected_recentralization_values
]
x = range(hybrid_performance.shape[1])
marker_positions = np.linspace(0, len(x) - 1, num=6, dtype=int)

fig, ax = plt.subplots()
set_axis_style(ax)

markers = ["s", "v", "o", "^", "D"]
for idx, marker in zip(selected_indices, markers):
    ax.plot(x, hybrid_performance[idx], "-" + marker,
            label=rf"$\beta$={recentralization_array[idx]:.1f}",
            markevery=marker_positions)

plt.xlabel('Time', fontweight='bold', fontsize=12)
plt.ylabel('Performance', fontweight='bold', fontsize=12)
ax.legend(frameon=False, ncol=1, fontsize=12, prop=legend_properties)
# savefig("hybrid_recentralization_performance_trajectories.png")
plt.show()
plt.clf()


In [ ]:
# Twin plot: final performance and diversity across re-centralization
fig, ax1 = plt.subplots()
ax1.set_xlabel(r'Re-centralization Degree ($\beta$)', fontweight='bold', fontsize=12)
ax1.set_ylabel('Final Performance', fontweight='bold', color="black", fontsize=12)

ax2 = ax1.twinx()
ax2.set_ylabel('Final Diversity', fontweight='bold', color="black", fontsize=12)

ax2.spines['left'].set_color(nature_orange)
ax1.yaxis.label.set_color(nature_orange)
ax1.tick_params(axis='y', colors=nature_orange)
ax2.spines['right'].set_color(nature_blue)
ax2.yaxis.label.set_color(nature_blue)
ax2.tick_params(axis='y', colors=nature_blue)

set_axis_style(ax1)
set_axis_style(ax2)

line_1 = ax1.plot(recentralization_array, final_value(hybrid_performance), "-s",
                  label="Performance", color=nature_orange)
line_2 = ax2.plot(recentralization_array, final_value(hybrid_diversity), "--v",
                  label="Diversity", color=nature_blue)

plt.xticks(recentralization_array)
lines = line_1 + line_2
labs = [line.get_label() for line in lines]
ax1.legend(lines, labs, frameon=False, fontsize=12, loc=(0, 1.02),
           ncol=2, prop=legend_properties)

savefig("hybrid_recentralization_performance_diversity.png")
plt.show()
plt.clf()
